In [17]:
# STEP 1: Import libraries and load dataset
# Description: Load required libraries and read the CSV file into a pandas DataFrame

import pandas as pd
import ast

file_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_cons_update5.csv"
df = pd.read_csv(file_path)

df.head()

,BUILDING_REFERENCE_NUMBER,OSG_REFERENCE_NUMBER,ADDRESS1,ADDRESS2,ADDRESS3,POSTCODE,LATITUDE,LONGITUDE,INSPECTION_DATE,TYPE_OF_ASSESSMENT,...,ROOF_CONS,WINDOW_CONS,DETACHED_2F_WINDOW_AREA_SINGLE,DETACHED_2F_WINDOW_H,DETACHED_2F_WINDOW_W,LIGHTING_LUM_PER_W,LIGHTING_LUMENS,LIGHTING_WATTS,NUMBER_OF_OCCUPANTS,OCCUPANTS_ROUNDED_UP
0,1001829067,122009933.0,19 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.055692,-4.223337,9/29/25,"RdSAP, existing dwelling",...,insulated_roof_300mm,dbl_glz_pre_2003,1.139618,0.871634,1.307451,66.9,7812,116.77,1.460450,2
1,1001951858,122010025.0,GLENVIEW,FINTRY,GLASGOW,G63 0XJ,56.052824,-4.220640,9/19/25,"RdSAP, existing dwelling",...,insulated_roof_300mm,dbl_glz_pre_2003,2.029782,1.163266,1.744899,66.9,23436,350.31,2.883810,3
2,1001829071,122009868.0,22 CULCREUCH AVENUE,FINTRY,GLASGOW,G63 0YB,56.055503,-4.223691,7/29/25,"RdSAP, existing dwelling",...,insulated_roof_200mm,dbl_glz_pre_2003,1.301291,0.931411,1.397117,80.5,12648,157.12,2.196607,3
3,1234761001,122009968.0,1 MENZIES TERRACE,FINTRY,GLASGOW,G63 0YJ,56.058427,-4.224838,7/22/25,"RdSAP, existing dwelling",...,insulated_roof_250mm,dbl_glz_pre_2003,2.097027,1.182378,1.773567,66.9,23622,353.09,2.886767,3
4,1001991633,122009892.0,50 MAIN STREET,FINTRY,GLASGOW,G63 0XF,56.054738,-4.228307,7/17/25,"RdSAP, existing dwelling",...,insulated_roof_200mm,dbl_glz_pre_2003,1.758582,1.082769,1.624153,66.9,19344,289.15,2.773598,3


In [18]:
# STEP 2: Define efficiency mapping
# Description: Map EPC efficiency grades (main_energy) to numeric suffix values

efficiency_map = {
    "very_poor": 1,
    "poor": 2,
    "average": 3,
    "good": 4,
    "very_good": 5
}

In [19]:
# STEP 3: Parse HEATING_JSON column
# Description: Convert stringified JSON into Python objects (lists/dicts)

def parse_heating_json(val):
    try:
        return ast.literal_eval(val)
    except:
        return None

df["HEATING_PARSED"] = df["HEATING_JSON"].apply(parse_heating_json)

In [20]:
# STEP 4: Extract heating fields
# Description: Extract main_heat, main_energy, and secondary_heat into separate columns

def extract_heating_fields(heating_list):
    if isinstance(heating_list, list) and len(heating_list) > 0:
        entry = heating_list[0]
        main_heat = entry.get("main_heat")
        main_energy = entry.get("main_energy")
        secondary_heat = entry.get("secondary_heat")
        return pd.Series([main_heat, main_energy, secondary_heat])
    else:
        return pd.Series([None, None, None])

df[["main_heat", "main_energy", "secondary_heat"]] = df["HEATING_PARSED"].apply(extract_heating_fields)

In [21]:
# STEP 5: Create MAIN_HEAT_CODE
# Description: Combine main_heat with efficiency suffix

def create_main_code(row):
    if pd.notnull(row["main_heat"]) and pd.notnull(row["main_energy"]):
        suffix = efficiency_map.get(row["main_energy"])
        if suffix:
            return f"{row['main_heat']}_{suffix}"
    return None

df["MAIN_HEAT_CODE"] = df.apply(create_main_code, axis=1)

In [22]:
# STEP 6 (UPDATED): Create SECOND_HEAT_CODE with "none" handling
# Description: If secondary_heat == "none", set SECOND_HEAT_CODE to "none".
# Otherwise, combine secondary_heat with efficiency suffix from main_energy.

def create_second_code(row):
    if pd.isnull(row["secondary_heat"]):
        return None
    
    if str(row["secondary_heat"]).lower() == "none":
        return "none"
    
    if pd.notnull(row["main_energy"]):
        suffix = efficiency_map.get(row["main_energy"])
        if suffix:
            return f"{row['secondary_heat']}_{suffix}"
    
    return None

df["SECOND_HEAT_CODE"] = df.apply(create_second_code, axis=1)

In [23]:
# STEP 7: Save updated dataset
# Description: Save the modified DataFrame back to a new CSV file

output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_with_heat_codes.csv"
df.to_csv(output_path, index=False)

print("File saved to:", output_path)

File saved to: /Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_with_heat_codes.csv


In [24]:
# STEP 8: Quick validation check
# Description: Display a few rows to confirm outputs look correct

df[["main_heat", "main_energy", "secondary_heat", "MAIN_HEAT_CODE", "SECOND_HEAT_CODE"]].head(10)

,main_heat,main_energy,secondary_heat,MAIN_HEAT_CODE,SECOND_HEAT_CODE
0,electric_storage_heaters,good,room_heaters_electric,electric_storage_heaters_4,room_heaters_electric_4
1,electric_storage_heaters,average,room_heaters_electric,electric_storage_heaters_3,room_heaters_electric_3
2,air_source_heat_pump_radiators_electric,good,room_heaters_dual_fuel,air_source_heat_pump_radiators_electric_4,room_heaters_dual_fuel_4
3,boiler_radiators_oil,average,none,boiler_radiators_oil_3,none
4,room_heaters_electric,very_poor,room_heaters_wood_logs,room_heaters_electric_1,room_heaters_wood_logs_1
5,room_heaters_electric,very_poor,none,room_heaters_electric_1,none
6,boiler_radiators_dual_fuel_mineral_wood,average,room_heaters_dual_fuel,boiler_radiators_dual_fuel_mineral_wood_3,room_heaters_dual_fuel_3
7,boiler_radiators_oil,average,room_heaters_dual_fuel,boiler_radiators_oil_3,room_heaters_dual_fuel_3
8,air_source_heat_pump_radiators_electric,good,room_heaters_dual_fuel,air_source_heat_pump_radiators_electric_4,room_heaters_dual_fuel_4
9,air_source_heat_pump_underfloor_electric,good,room_heaters_wood_logs,air_source_heat_pump_underfloor_electric_4,room_heaters_wood_logs_4
